# Train Model Ensemble (N=100)

This notebook trains ensembles of 100 MPNN models using the best hyperparameters identified in `02_model_tuning.ipynb`.

**Approach:**
1. Load best hyperparameters from tuning results (separately for with/without descriptors)
2. Train 100 models with different random seeds for each configuration
3. Save all models to disk for downstream analysis

**Two ensembles:**
- **With descriptors**: Uses best config from tuning where `use_descriptors=True`
- **Without descriptors**: Uses best config from tuning where `use_descriptors=False`

In [ ]:
# import necessary libraries
import sys
import os
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from chemprop import data, featurizers, models, nn, utils
from chemprop.models.utils import save_model, load_model
from chemprop.nn.metrics import RMSE

In [ ]:
# Ensure project root is on path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
figs_dir = PROJECT_ROOT / "figs"

models_dir = PROJECT_ROOT / "models"

In [ ]:
# reducing verbosity
import logging
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
import warnings
warnings.filterwarnings("ignore")

## Constants and Configuration

In [ ]:
# Data columns
smiles_column = 'SMILES_clean'
target_column = ['logTg']

# List of molecular descriptors
descriptor_list = [
    'Moleculer Weight',
    'Proxy for Free Volume',
    'Rotatable Bonds',
    'Aromatic Rings',
    'Topological Polar Surface Area',
    'H-Bond Donors',
    'H-Bond Acceptors'
]

In [ ]:
# Ensemble configuration
N_ENSEMBLE = 100  # Number of models in ensemble
DATA_SPLIT_SEED = 42   # Fixed seed for train/val split (consistent with tuning)

# Training configuration (from tuning)
MAX_EPOCHS = 100
EARLY_STOP_PATIENCE = 20  # Increased patience to handle validation noise
NUM_WORKERS = 0

LOSS_CRITERION = RMSE() # Loss function
BATCH_SIZE = 64 # Batch size

K_FOLDS = 10

In [ ]:
# Generate unique random seeds for each ensemble member
train_seeds = np.random.default_rng(12345).integers(low=0, high=2**32, size=N_ENSEMBLE)
len(train_seeds)

## Prepare dataset

In [ ]:
# Load training data (used for validation set)
df_train = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "training_data.csv")

all_smis = df_train.loc[:, smiles_column].values
all_ys = df_train.loc[:, target_column].values  # Shape: (N,)
all_descriptors = df_train.loc[:, descriptor_list].values  # Shape: (N, num_descriptors)
print(f"All data: {len(all_smis)} total samples")
print(f"All targets shape: {all_ys.shape}")
print(f"All descriptors shape: {all_descriptors.shape}")

In [ ]:
# create RDKit molecule objects from SMILES notation strings
all_mols = [
    utils.make_mol(smi, keep_h=False, add_h=False) for smi in all_smis
]

In [ ]:
# KFold splitting
kf = KFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=DATA_SPLIT_SEED
)

# create dictionaries to hold train and validation indices for each fold
train_idx_dict = {}
val_idx_dict = {}
for fold, (train_idx, val_idx) in enumerate(kf.split(all_mols)):
    print(f"Fold {fold+1}")
    train_idx_dict[fold] = train_idx
    val_idx_dict[fold] = val_idx

In [ ]:
os.makedirs(models_dir, exist_ok=True)

In [ ]:
# save data indexes
with open(PROJECT_ROOT/ "models" / "train_data_split_indexes.pkl", "wb") as f:
    pickle.dump(train_idx_dict, f)

with open(PROJECT_ROOT / "models" / "validation_data_split_indexes.pkl", "wb") as f:
    pickle.dump(val_idx_dict, f)

# Train Ensemble of 100 Models

Train separate ensembles for with/without descriptors using best hyperparameters.

Each model uses:
- **K-Fold split** with K=5 for robustness across training data selection 
- **Different training seed** (for weight initialization, dropout randomness, etc.)
- **Early stopping** with patience=20

In [ ]:
# featurizer to transforms molecules into molecular graphs where atoms become nodes and bonds become edges
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

In [ ]:
def train_single_model_with_early_stopping(
    fold,
    train_idx,
    val_idx,
    all_mols,
    all_ys,
    configs,
    featurizer,
    logs_dir,
    train_seed,
    all_V_fs=None,
):
    """"""
    # Set seed for reproducibility
    pl.seed_everything(train_seed, workers=True, verbose=False)

    # Prepare data
    if all_V_fs is not None:
        train_data = [
            data.MoleculeDatapoint(
                all_mols[train_idx[i]],
                all_ys[train_idx[i]],
                x_d=all_V_fs[train_idx[i]]
            ) for i in range(len(train_idx))
        ]
        val_data = [
            data.MoleculeDatapoint(
                all_mols[val_idx[i]],
                all_ys[val_idx[i]],
                x_d=all_V_fs[val_idx[i]]
            ) for i in range(len(val_idx))
        ]
    else:
        train_data = [
            data.MoleculeDatapoint(
                all_mols[train_idx[i]],
                all_ys[train_idx[i]]
            ) for i in range(len(train_idx))
        ]
        val_data = [
            data.MoleculeDatapoint(
                all_mols[val_idx[i]],
                all_ys[val_idx[i]],
            ) for i in range(len(val_idx))
        ]

    train_dset = data.MoleculeDataset(train_data, featurizer)
    val_dset = data.MoleculeDataset(val_data, featurizer)

    fold_scaler = train_dset.normalize_targets()

    val_dset.normalize_targets(fold_scaler)

    if all_V_fs is not None:
        fold_desc_scaler = train_dset.normalize_inputs("X_d")
        val_dset.normalize_inputs("X_d", fold_desc_scaler)
    else:
        fold_desc_scaler = None

    train_loader = data.build_dataloader(
        train_dset,
        batch_size=configs["batch_size"],
        num_workers=configs["num_workers"],
        shuffle=True
    )
    val_loader = data.build_dataloader(
        val_dset,
        batch_size=configs["batch_size"],
        num_workers=configs["num_workers"],
        shuffle=False
    )

    # Model
    mp = nn.BondMessagePassing()
    
    agg = nn.MeanAggregation()
    
    output_transform = nn.UnscaleTransform.from_standard_scaler(fold_scaler)
    
    if all_V_fs is not None:
        ffn_input_dim = mp.output_dim + all_V_fs.shape[1]
        X_d_transform = nn.ScaleTransform.from_standard_scaler(fold_desc_scaler)
    else:
        ffn_input_dim = mp.output_dim
        X_d_transform = None

    ffn = nn.RegressionFFN(
        input_dim=ffn_input_dim,
        output_transform=output_transform,
        criterion=configs["loss_criterion"],
    )
    mpnn = models.MPNN(
        mp,
        agg,
        ffn,
        batch_norm=configs["batch_norm"],
        X_d_transform=X_d_transform
    )
    model_log_dir = Path(logs_dir) / f"fold_{fold+1}" / f"seed_{train_seed}"
    # Logging and checkpointing    
    csv_logger = pl.loggers.CSVLogger(
        save_dir=model_log_dir,
        name=f"fold_{fold+1}_seed_{train_seed}"
    )
    checkpoint_cb = ModelCheckpoint(
        dirpath=model_log_dir,
        filename="best_model",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        save_last=False,
    )
    early_stop_cb = EarlyStopping(
        monitor="val_loss",
        patience=configs['early_stop_patience'],
        mode="min",
        verbose=True,
    )

    trainer = pl.Trainer(
        logger=csv_logger,
        callbacks=[early_stop_cb, checkpoint_cb],
        enable_checkpointing=True,
        enable_progress_bar=False,
        accelerator="auto",
        devices=1,
        max_epochs=configs["max_epochs"],
        deterministic=configs["deterministic"],
        enable_model_summary=False,
        log_every_n_steps=999999
    )

    trainer.fit(
        mpnn,
        train_loader,
        val_loader
    )

    # Save info for reproducibility
    metrics_path = Path(trainer.logger.log_dir) / "metrics.csv"
    checkpoint_path = checkpoint_cb.best_model_path

    return {
        "fold": fold+1,
        "train_seed": train_seed,
        "metrics_csv": str(metrics_path),
        "checkpoint_path": str(checkpoint_path),
        "fold_scaling_factor": fold_scaler.scale_[0],
    }

In [ ]:
def train_ensemble_kfold(
    K_FOLDS,
    all_mols,
    all_ys,
    train_idx_dict,
    val_idx_dict,
    configs,
    featurizer,
    logs_dir,
    all_V_fs=None,
):
    ensemble_results = {}
    for fold in range(K_FOLDS):
        ensemble_results[fold] = []
        for train_seed in configs["train_seed_list"]:
            result = train_single_model_with_early_stopping(
                fold=fold,
                train_idx=train_idx_dict[fold],
                val_idx=val_idx_dict[fold],
                all_mols=all_mols,
                all_ys=all_ys,
                configs=configs,
                featurizer=featurizer,
                logs_dir=logs_dir,
                train_seed=train_seed,
                all_V_fs=all_V_fs,
            )
            
            ensemble_results[fold].append(result)
    
    return ensemble_results

In [ ]:
configs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "max_epochs": MAX_EPOCHS,
    "loss_criterion": LOSS_CRITERION,
    "batch_norm": True,
    "deterministic": True,
    "early_stop_patience": EARLY_STOP_PATIENCE,
}

### Train Ensemble WITH Descriptors

In [ ]:
print(f"Training {N_ENSEMBLE} models WITH descriptors using K-Fold and early stopping...")

ensemble_with_desc = []
history_with_desc = []

with_desc_dir = PROJECT_ROOT / "models" / "ensemble_with_descriptors"
os.makedirs(with_desc_dir, exist_ok=True)

for fold_ in range(K_FOLDS):
    print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
    for i in tqdm(range(N_ENSEMBLE), desc="Training WITH descriptors"):
        train_seed = int(train_seeds[i])
        result = train_single_model_with_early_stopping(
            fold=fold_,
            train_idx=train_idx_dict[fold],
            val_idx=val_idx_dict[fold],
            all_mols=all_mols,
            all_ys=all_ys,
            configs=configs,
            featurizer=featurizer,
            logs_dir=with_desc_dir,
            train_seed=train_seed,
            all_V_fs=all_descriptors,
        )
        # Save checkpoint/model path and training history
        ensemble_with_desc.append(result["checkpoint_path"])
        
        # Optionally, load and store loss curves for later analysis
        metrics_df = pd.read_csv(result["metrics_csv"])
        metrics_df = metrics_df.sort_values(['epoch', 'step'])
        metrics_df_epoch = metrics_df.groupby('epoch', as_index=False).last()
        best_epoch = metrics_df_epoch['val_loss'].idxmin()
        best_epoch_number = metrics_df_epoch.iloc[best_epoch]['epoch']
        
        history_with_desc.append({
            "train_loss": metrics_df_epoch["train_loss_epoch"].tolist(),
            "val_loss": metrics_df_epoch["val_loss"].tolist(),
            "train_loss_unscaled": (metrics_df_epoch["train_loss_epoch"] * result["fold_scaling_factor"]).tolist(),
            "val_loss_unscaled": (metrics_df_epoch["val_loss"] * result["fold_scaling_factor"]).tolist(),
            "fold": result["fold"],
            "train_seed": result["train_seed"],
            "scaling_factor": result["fold_scaling_factor"],
            "best_epoch": best_epoch_number,
        })

print(f"✓ Trained {len(ensemble_with_desc)} models WITH descriptors")

### Train Ensemble WITHOUT Descriptors

In [ ]:
# Train ensemble WITHOUT descriptors
print(f"Training {N_ENSEMBLE} models WITHOUT descriptors...")

ensemble_no_desc = []
history_no_desc = []

no_desc_dir = PROJECT_ROOT / "models" / "ensemble_without_descriptors"
os.makedirs(no_desc_dir, exist_ok=True)
for fold_ in range(K_FOLDS):
    print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
    for i in tqdm(range(N_ENSEMBLE), desc="Training WITHOUT descriptors"):
        train_seed = int(train_seeds[i])
        result = train_single_model_with_early_stopping(
            fold=fold_,
            train_idx=train_idx_dict[fold],
            val_idx=val_idx_dict[fold],
            all_mols=all_mols,
            all_ys=all_ys,
            configs=configs,
            featurizer=featurizer,
            logs_dir=no_desc_dir,
            train_seed=train_seed,
        )
        # Save checkpoint/model path and training history
        ensemble_no_desc.append(result["checkpoint_path"])
        
        # Optionally, load and store loss curves for later analysis
        metrics_df = pd.read_csv(result["metrics_csv"])
        metrics_df = metrics_df.sort_values(['epoch', 'step'])
        metrics_df_epoch = metrics_df.groupby('epoch', as_index=False).last()
        
        history_no_desc.append({
            "train_loss": metrics_df_epoch["train_loss_epoch"].tolist(),
            "val_loss": metrics_df_epoch["val_loss"].tolist(),
            "train_loss_unscaled": (metrics_df_epoch["train_loss_epoch"] * result["fold_scaling_factor"]).tolist(),
            "val_loss_unscaled": (metrics_df_epoch["val_loss"] * result["fold_scaling_factor"]).tolist(),
            "fold": result["fold"],
            "train_seed": result["train_seed"],
            "scaling_factor": result["fold_scaling_factor"],
        })

print(f"✓ Trained {len(ensemble_no_desc)} models WITHOUT descriptors")

## Ensemble Training Loss Visualization

Visualize training and validation losses across the ensemble to assess convergence and consistency.

In [ ]:
def make_results_dict_from_history(history_list):
    results_dict = {}
    for h in history_list:
        fold = h['fold']
        
        if fold not in results_dict:
            results_dict[fold] = []
        
        results_dict[fold].append({
            'train_loss_epoch': h['train_loss'],
            'val_loss': h['val_loss'],
            'fold_scaling_factor': h['scaling_factor'],
        })
    return results_dict

In [ ]:
def plot_all_learning_curves(history_list, title_suffix="", save_path=None):
    plt.figure(figsize=(7, 5))
    for h in history_list:
        plt.plot(h['train_loss_unscaled'], color='black')
        plt.plot(h['val_loss_unscaled'], color='gray')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('RMSE (original units)', fontsize=12)
    plt.title(f'All Learning Curves {title_suffix}', fontsize=12)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, alpha=0.2)
    plt.legend(['Training', 'Validation'], fontsize=12)
    if save_path:
        plt.savefig(save_path, dpi=400, bbox_inches='tight')
    plt.show()

In [ ]:
# Usage:
plot_all_learning_curves(
    history_with_desc,
    title_suffix="Ensemble WITH Descriptors",
    save_path=figs_dir / "ensemble_w_desc_learning_curve.png"
)

In [ ]:
# Usage:
plot_all_learning_curves(
    history_no_desc,
    title_suffix="Ensemble WITHOUT Descriptors",
    save_path=figs_dir / "ensemble_no_desc_learning_curve.png"
)

In [ ]:
# Collect best (minimum) train and val RMSE for each run in the ensemble

def get_best_losses(history_list):
    best_train = []
    best_val = []
    for h in history_list:
        best_val_idx = np.argmin(h['val_loss_unscaled'])
        best_val.append(h['val_loss_unscaled'][best_val_idx])
        best_train.append(h['train_loss_unscaled'][best_val_idx])
    return np.array(best_train), np.array(best_val)

In [ ]:
best_train_with, best_val_with = get_best_losses(history_with_desc)
best_train_no, best_val_no = get_best_losses(history_no_desc)

print(f"WITH descriptors: Mean best train RMSE: {best_train_with.mean():.4f}")
print(f"WITH descriptors: Mean best val RMSE: {best_val_with.mean():.4f}")
print(f"WITHOUT descriptors: Mean best train RMSE: {best_train_no.mean():.4f}")
print(f"WITHOUT descriptors: Mean best val RMSE: {best_val_no.mean():.4f}")

In [ ]:
# Plot histograms of best losses for each ensemble (train/val, with/without descriptors)
loss_data = [
    (best_train_with, 'WITH Descriptors - Best Train RMSE', 'gray'),
    (best_val_with,   'WITH Descriptors - Best Val RMSE',   'gray'),
    (best_train_no,   'WITHOUT Descriptors - Best Train RMSE', 'gray'),
    (best_val_no,     'WITHOUT Descriptors - Best Val RMSE',   'gray'),
]

fig, axes = plt.subplots(2, 2, figsize=(7, 6))
for idx, (data, title, color) in enumerate(loss_data):
    row, col = divmod(idx, 2)
    axes[row, col].hist(data, bins=20, color=color, alpha=0.7, edgecolor='black')
    axes[row, col].axvline(np.mean(data), color='black', linestyle='--', linewidth=2, label=f'Mean: {np.mean(data):.4f}')
    axes[row, col].set_title(title)
    axes[row, col].set_xlabel('RMSE')
    axes[row, col].set_ylabel('Frequency')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.suptitle('Histogram of Best Losses (per early-stopped model)', fontsize=12)
plt.tight_layout()
plt.savefig(figs_dir / "ensemble_best_loss_histogram.eps", dpi=300, bbox_inches='tight')
plt.show()